1.b

Código do projeto anterior na 5.a:

In [ ]:
import numpy as np

def movimentoAntigo(N, A, x0, v0, t, k=1.0, m=1.0):
    autovalores, autovetores = autovaloresAutovetores(A)

    a = np.dot(autovetores.T, x0.T)
    b = np.dot(autovetores.T, v0.T)
    
    x_t = np.zeros((N, len(t)))
    v_t = np.zeros((N, len(t)))

    for i in range(N):
        lmb = autovalores[i]
        
        if lmb < 1e-10:
            for j in range(len(t)):
                q_t = a[i] + b[i] * t[j]
                dq_t = b[i]
                x_t[:, j] += autovetores[:, i] * q_t
                v_t[:, j] += autovetores[:, i] * dq_t
        else:
            w = np.sqrt(lmb * (k / m))
            for j in range(len(t)):
                q_t = a[i] * np.cos(w * t[j]) + (b[i] / w) * np.sin(w * t[j])
                dq_t = -a[i] * w * np.sin(w * t[j]) + b[i] * np.cos(w * t[j])
                x_t[:, j] += autovetores[:, i] * q_t
                v_t[:, j] += autovetores[:, i] * dq_t

    return x_t, v_t

def matrizSegundaLei(N, kList, k):
  A0 = np.zeros((N, N))
  V = np.zeros((N, N))
  Mid = np.eye(N)  

  for i in range(N):
    for j in range(N):

      A0[i][j] = (
          2 * Mid[i][j] - (Mid[i][j + 1] if j + 1 < N else 0) - (Mid[i][j - 1] if j - 1 >= 0 else 0) - Mid[i][0] * Mid[j][0] - Mid[i][N - 1] * Mid[j][N - 1]
      )
      
      V[i][j] = (kList[i] / k) * Mid[i][j]

  A = A0 + V
  return A

def autovaloresAutovetores(A):
    eigenvalues, eigenvectors = np.linalg.eigh(A)
    
    return eigenvalues, eigenvectors

1.b

Código adaptado para receber, também, $\vec{u}$ e retornar $\vec{x}$ e $\vec{v}$.

In [ ]:
def movimento(N, A, x0, v0, u, t, k=1.0, m=1.0):

    y0 = x0 - u
    
    autovalores, autovetores = autovaloresAutovetores(A)
    
    a = np.dot(autovetores.T, y0.T)
    b = np.dot(autovetores.T, v0.T) # a velocidade n muda pois se derivar uma vez o u some
    
    y_t = np.zeros((N, len(t)))
    v_t = np.zeros((N, len(t)))

    for i in range(N):
        lmb = autovalores[i]
        
        if lmb < 1e-10:
            for j in range(len(t)):
                q_t = a[i] + b[i] * t[j]
                dq_t = b[i]
                y_t[:, j] += autovetores[:, i] * q_t
                v_t[:, j] += autovetores[:, i] * dq_t
        else:
            w = np.sqrt(lmb * (k / m))
            for j in range(len(t)):
                q_t = a[i] * np.cos(w * t[j]) + (b[i] / w) * np.sin(w * t[j])
                dq_t = -a[i] * w * np.sin(w * t[j]) + b[i] * np.cos(w * t[j])
                y_t[:, j] += autovetores[:, i] * q_t
                v_t[:, j] += autovetores[:, i] * dq_t

    return y_t + u.reshape(N,1), v_t

def montarAr(N, kmuro, k = 1.0):
    A0 = matrizSegundaLei(N, np.zeros(N), k)
    xNxNT = np.zeros((N,N))
    xNxNT[N-1][N-1] = 1
    
    Ar = A0 + kmuro/k * (xNxNT)